# M3C4 — Level 0: Feature Selection, Explained 🌱

**Purpose:** understand why we compare feature sets and follow one complete example.

**Do:** save a copy in Drive and run each code cell with Shift + Enter. When asked, upload both `telco_source.csv` and `M3C4_Runtime.zip`. In local Jupyter keep them beside this notebook. After a reset, start again at C01.

**Check:** read the output after every step and write your own C04 and C16 notes. This example chooses `top_k = 10` before validation. The final test stays closed.

## What are we trying to predict?

Each row describes a customer. A **feature** is an input, such as tenure or a contract category. The **target** is the known `Churn` answer: Yes means the customer left. We learn patterns on the training rows, compare choices on separate validation rows, and keep the final test closed.

Last lesson created five additional features. Today we ask whether a smaller set can be useful. More inputs do not automatically make a better model. We will inspect overlap, rank candidates, and compare results on the same validation customers.

## C01 — 🚀 Prepare the workspace

**Purpose:** rebuild the previous lesson's features and the protected split. The setup code is ready; you do not need to edit it.

**Do:** run the collapsed code cell. Select both files if a picker opens.

**Check:** the output shows 4,225 train rows, 1,409 validation rows, 1,409 reserved test rows, 50 prepared columns and five training rows without IDs. The test is reserved, not used to choose a feature set.

In [ ]:
#@title ▶ Start the lesson
from pathlib import Path
from zipfile import ZipFile, BadZipFile
comparison = results = rfe_columns = None  # Invalidate old results before a new start.
required = ['telco_source.csv', 'M3C4_Runtime.zip']
if any(not Path(name).exists() for name in required):
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Place telco_source.csv and M3C4_Runtime.zip beside this notebook, then rerun C01.') from None
    print('Select BOTH telco_source.csv and M3C4_Runtime.zip from the lesson package.')
    files.upload()
missing = [name for name in required if not Path(name).exists()]
if missing:
    raise FileNotFoundError('Missing: ' + ', '.join(missing) + '. Rerun C01 with the issued files and original names.')
try:
    with ZipFile('M3C4_Runtime.zip') as support:
        folder = Path('_m3c4_runtime'); folder.mkdir(exist_ok=True)
        for name in ['m3c4_runtime.py', 'M3_Split_Card.json']:
            (folder / name).write_bytes(support.read(name))
except (BadZipFile, KeyError):
    raise ValueError('Wrong or damaged M3C4_Runtime.zip. Upload the issued ZIP again, then rerun C01.') from None
exec(compile((folder / 'm3c4_runtime.py').read_text(encoding='utf-8'), 'm3c4_runtime.py', 'exec'), globals())

## C02–C03 — 🔎 Correlation: do two columns move together?

**Purpose:** find numeric inputs that may repeat similar information. Pearson correlation **r** measures a straight-line relationship from −1 to +1. A value near +1 means both rise together; near −1 means one tends to fall as the other rises. A value near zero does not rule out a curved relationship.

**Small example:** tenure in months `[0, 12, 24, 36]` and years `[0, 1, 2, 3]` describe the same time. Predict their relationship before running. The declared order gives tenure priority over tenure_years.

**Do:** run the ready code. It uses training data only. `abs(r)` ignores the sign; the rule checks strictly **abs(r) > 0.85**, so exactly 0.85 is not included. The prepared rule keeps the earlier field in its declared source/formula order; a pair already covered by a removal needs no second deletion.

**Check:** inspect the toy table, then find an actual high-correlation pair and its removal record. The `all` model later still uses the full 50-column set; the filtered candidates feed MI and RFE.

In [ ]:
toy=pd.DataFrame({'tenure':[0,12,24,36],'tenure_years':[0,1,2,3],'fee':[30,50,20,40]})
show_table('Toy example · identical information in different units','Tenure and tenure_years move together exactly in this toy example.',toy.corr().rename_axis('Feature').reset_index())
assert toy.corr().loc['tenure','tenure_years']==1
correlation_report()

## C04 — Your correlation notes ✍️

**Do:** use an actual pair and its removal record from C03. Write in your own words.

**Pair and r:** …

**Removed field and why the other is kept:** …

**What this filter cannot prove:** …

**Check:** correlation describes a relationship; it does not establish a cause.

## C08 — 📊 Mutual Information

**Purpose:** rank each candidate by information it provides about the target. **Mutual Information (MI)** can detect relationships that are not straight lines. A high individual score does not guarantee the best group of features.

**Small example:** four customers have Churn `[Yes, Yes, No, No]`. A flag `[1, 1, 0, 0]` separates these outcomes; a constant `[1, 1, 1, 1]` does not. Which would you expect to be more informative?

One-hot turns a category into separate yes/no columns. Discrete values are separate categories or counts (such as 0/1 or number of services); continuous values represent measurements (such as charges).

**Do:** run the ready train-only ranking.

**Check:** read the candidate count and discrete-column list. One-hot columns, the SeniorCitizen category and the num_services count are discrete; the other numeric fields are continuous. A name like `cat__Contract_Month-to-month` asks whether the contract is month-to-month.

In [ ]:
# C08 — Prepared train-only MI ranking. No validation results yet.
mi_ranking()

## C09 — Choose a count before validation

**Purpose:** turn the ranking into a feature set. `top_k` is the **number of top-ranked features**, not an MI-score threshold.

**Do:** this worked example fixes **10** before looking at validation results. Run the cell. This is a demonstration choice, not a claim that 10 is best.

**Check:** later, compare its result with the full set and RFE. If you change the choice or learning code, rerun C09, C11, C12, C14 and C15 in order. Do not repeatedly tune on validation.

In [ ]:
top_k = 10
selection_reason = 'Worked example: choose 10 before viewing validation results.'

## Before C11 — How will we read the results?

A prediction of **Yes** means the model predicts a customer will leave. Imagine 10 customers: the model predicts Yes for four; three of those four actually leave. Five customers actually leave in total.

**Accuracy** is correct predictions divided by all predictions. **Precision** asks: of the four predicted leavers, how many actually left? Here it is 3/4. **Recall** asks: of the five actual leavers, how many did we find? Here it is 3/5. **F1** combines precision and recall using their harmonic mean: `2 × precision × recall / (precision + recall)`. Both matter when missing a leaver and raising a false alarm have different costs.

The output below reports Accuracy and F1, not separate precision and recall. These toy numbers explain the terms; they are not results from Telco.

## C11–C12 — ✅ Compare all and MI

**Purpose:** see how two input sets perform using the same model and validation customers. The ready code fits a Random Forest with 100 trees and seed 42 for each set.

**Do:** run both cells in order. The second independently checks the first function's results.

**Check:** read the two labelled rows, feature counts, Accuracy and F1. These are results from your own run, not promises about other data.

In [ ]:
# Provided learning code — run this cell.
def compare_all_mi(X_train, X_validation, y_train, y_validation, mi_columns):
    rows=[]
    for label, columns in [('all',list(X_train.columns)),('MI',list(mi_columns))]:
        model=RandomForestClassifier(n_estimators=100,random_state=42)
        model.fit(X_train[columns],y_train)
        predictions=model.predict(X_validation[columns])
        rows.append({'Feature Set':label,'Num Features':len(columns),'Accuracy':accuracy_score(y_validation,predictions),'F1':f1_score(y_validation,predictions,pos_label='Yes')})
    return pd.DataFrame(rows)

In [ ]:
# Intermediate all versus MI; RFE joins in C15.
validate_compare()

## C14 — 🔁 Recursive Feature Elimination

**Purpose:** use a model to choose a subset by repeatedly removing one input. **RFE** means Recursive Feature Elimination.

**Small example:** suppose we want two of four inputs. Fit a model with all four and remove the lowest-ranked input: **4 → 3**. Fit again on the remaining three and remove the lowest-ranked: **3 → 2**. The rankings depend on the model and the data; this is an illustration, not a Telco result.

**Do:** run the prepared selector. It uses the filtered training candidates, Random Forest with 100 trees and seed 42, removes one at a time, and keeps half the candidates rounded down (at least one). It can take a little time.

**Check:** `Selected=True` and rank 1 identify the kept features. Read the runtime and MI ∩ RFE: the features selected by both methods.

In [ ]:
# C14 — Prepared RFE: RF100 seed42, step=1, floor(half candidates), at least 1.
# https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFE.html
run_rfe()

## C15 — 📋 Compare the three sets

**Purpose:** compare all 50 features, MI's 10 and the RFE selection on the same validation rows.

**Do:** run the table. Read feature count, Accuracy and F1 together. The next note translates the accuracy gap into a number of predictions among these 1,409 customers.

**Check:** find all three labelled rows. A small difference on this validation set is not proof of a general winner or a causal relationship.

In [ ]:
# Same model and validation customers for all three sets.
final_table()

## C16 — Your recommendation ✍️

**Do:** use the actual table and RFE output; write your own answers.

**I recommend:** …

**Actual Accuracy, F1 and feature counts supporting my choice:** …

**MI/RFE overlap and one disagreement, if any:** …

**How many validation predictions does the accuracy gap represent?** …

**What can this evidence say about MonthlyCharges and churn? What can it not prove?** …

**For new training data, what must be learned again for preparation, correlation, MI and RFE? When would you choose top_k?** …

**Check:** selection for prediction does not prove a feature causes churn. Your own analysis needs human review.

## Save and reopen

**Do:** save the notebook in Drive or download the `.ipynb`. Reopen it and check your C04 and C16 notes and current C15 table.

**Check:** after changing the choice or code, rerun C09, C11, C12, C14 and C15. After a runtime reset, begin at C01; old outputs can be stale. The final test remains closed.